## Importing and Setup


In [1]:
# Install libraries from inside the notebook
%pip install polars pandas numpy faker plotly scikit-learn xgboost streamlit --quiet

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Importing required libraries for data handling and random data generation
import polars as pl
import pandas as pd
import numpy as np
import random
from pathlib import Path
from faker import Faker

# Setup Faker to generate Indian names and cities
faker = Faker("en_IN")

# Define the data folder path (already created)
# This insures CSV are stored in the correct folder automatically
data_folder = Path(
    "/Users/Mudit PC/OneDrive/Desktop/Data analytics projects/Indian_Beauty_Analytics/data/"
)

## Generate Product Data


In [3]:
# Define a list of products with product_id, name, category and price
products = [
    {"product_id": f"p{i}", "product_name": name, "category": cat, "price": price}
    for i, (name, cat, price) in enumerate(
        [
            ("Aloe Vera Gel", "Skincare", 299),
            ("Vitamin C Serum", "Skincare", 499),
            ("Herbal Shampoo", "Haircare", 199),
            ("Lip Balm", "Makeup", 149),
            ("Face Wash", "Skincare", 399),
            ("Sunscreen SPF50", "Skincare", 599),
            ("Hair Oil", "Haircare", 249),
            ("Body Lotion", "Skincare", 349),
            ("Nail Polish", "Makeup", 199),
            ("Face Pack", "Skincare", 299)
        ]
    )
]

# Convert the product list into a polars DataFrame
products_df = pl.DataFrame(products)

# Save the products DataFrame to csv inside the data folder
products_df.write_csv(data_folder / "products.csv")

# Display the first few rows of the products DataFrame
products_df.head()

product_id,product_name,category,price
str,str,str,i64
"""p0""","""Aloe Vera Gel""","""Skincare""",299
"""p1""","""Vitamin C Serum""","""Skincare""",499
"""p2""","""Herbal Shampoo""","""Haircare""",199
"""p3""","""Lip Balm""","""Makeup""",149
"""p4""","""Face Wash""","""Skincare""",399


## Generate Customer Data


In [4]:
# Set the number of customers to generate
num_customers = 1000

# Initialize an emply list to store customer dictionaries
customers = []

# Loop to generate each customer
for i in range(num_customers):
    customers.append({
        "customer_id": f"c{i}",  # Unique Customer ID
        "name": faker.name(),    # Random Indian Customer Name
        "city": faker.city(),   # Random Indian City
        "age": random.randint(18, 55),  # Random Age between 18 and 55
        "gender": random.choice(["Male", "Female"]),  # Random Gender
    }
    )

# Convert the customer list into a polars DataFrame
customers_df = pl.DataFrame(customers)

# Save the customers DataFrame to csv inside the data folder
customers_df.write_csv(data_folder / "customers.csv")

# Display the first few rows of the customers DataFrame
customers_df.head()

customer_id,name,city,age,gender
str,str,str,i64,str
"""c0""","""Brijesh Munshi""","""Bikaner""",26,"""Male"""
"""c1""","""Tanveer Kade""","""Bhubaneswar""",43,"""Male"""
"""c2""","""Hamsini Tripathi""","""Anantapuram""",42,"""Male"""
"""c3""","""Ekantika Chada""","""Sasaram""",27,"""Female"""
"""c4""","""Jeet Shere""","""Gaya""",32,"""Male"""


## Generate Transactions


In [ ]:
# Set the number of transactions to generate
num_transactions = 50000

# Initialize an empty list to store transaction dictionaries
transactions = []

# Define festival periods with start and end dates
festivals = {
    "Diwali": ("2024-10-10", "2024-11-10"),
    "Holi": ("2024-03-01", "2024-03-31"),
    "Summer": ("2024-05-01", "2024-06-30")
}

# Helper function to pick a random date between start and end


def random_date(start, end):
    return pd.to_datetime(np.random.choice(pd.date_range(start, end)))


# Loop to generate each transaction
for i in range(num_transactions):
    # Randomly pick a customer
    customer = customers_df.sample(1).to_dict(as_series=False)

    # Randomly pick a product
    product = products_df.sample(1).to_dict(as_series=False)

    # 20% chance the purchase happens during a festival
    if random.random() < 0.2:
        fest = random.choice(list(festivals.keys()))
        purchase_date = random_date(festivals[fest][0], festivals[fest][1])
        festival = fest
    else:
        # Non-festival purchase
        purchase_date = random_date("2024-01-01", "2024-12-31")
        festival = None

    # Add the transaction record to the list
    transactions.append({
        "transaction_id": f"T{i}",                    # Unique transaction ID
        "customer_id": customer["customer_id"][0],   # Customer ID
        "product_id": product["product_id"][0],      # Product ID
        "purchase_date": purchase_date,              # Purchase date
        "price": product["price"][0],                # Product price
        "city": customer["city"][0],                 # Customer city
        "festival": festival                          # Festival name or None
    })

# Convert the list of transactions into a Polars DataFrame
transactions_df = pl.DataFrame(transactions)

# Save the transactions DataFrame to CSV inside the data folder
transactions_df.write_csv(data_folder / "transactions.csv")

# Display the first few rows of the transactions DataFrame
transactions_df.head()

transaction_id,customer_id,product_id,purchase_date,price,city,festival
str,str,str,datetime[μs],i64,str,str
"""T0""","""c1""","""p4""",2024-09-23 00:00:00,399,"""Bhubaneswar""",null
"""T1""","""c948""","""p1""",2024-11-12 00:00:00,499,"""Ghaziabad""",null
"""T2""","""c802""","""p6""",2024-03-04 00:00:00,249,"""Serampore""",null
"""T3""","""c525""","""p9""",2024-09-02 00:00:00,299,"""Ghaziabad""",null
"""T4""","""c488""","""p6""",2024-08-06 00:00:00,249,"""Purnia""",null


## Quick Exploration


In [8]:
# Display top products by number of transactions
print("Top Products by Sales")

print(
    transactions_df
    .group_by("product_id")
    .agg(
        pl.count().alias("transaction_count")
    )
    .sort("transaction_count", descending=True)
    .head(5)
)


# Display cities with the most transactions
print("\nTop Cities by Transactions")

print(
    transactions_df
    .group_by("city")
    .agg(
        pl.count().alias("transaction_count")
    )
    .sort("transaction_count", descending=True)
    .head(5)
)


# Display number of purchases during festivals
print("\nFestival Purchases")

print(
    transactions_df
    .group_by("festival")
    .agg(
        pl.count().alias("transaction_count")
    )
    .sort("transaction_count", descending=True)
)

Top Products by Sales
shape: (5, 2)
┌────────────┬───────────────────┐
│ product_id ┆ transaction_count │
│ ---        ┆ ---               │
│ str        ┆ u32               │
╞════════════╪═══════════════════╡
│ p9         ┆ 5285              │
│ p8         ┆ 5234              │
│ p5         ┆ 5056              │
│ p0         ┆ 5028              │
│ p4         ┆ 5015              │
└────────────┴───────────────────┘

Top Cities by Transactions
shape: (5, 2)
┌──────────────┬───────────────────┐
│ city         ┆ transaction_count │
│ ---          ┆ ---               │
│ str          ┆ u32               │
╞══════════════╪═══════════════════╡
│ South Dumdum ┆ 446               │
│ Allahabad    ┆ 435               │
│ Berhampore   ┆ 421               │
│ Haldia       ┆ 411               │
│ Ghaziabad    ┆ 406               │
└──────────────┴───────────────────┘

Festival Purchases
shape: (4, 2)
┌──────────┬───────────────────┐
│ festival ┆ transaction_count │
│ ---      ┆ ---              

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_24040\1013134287.py:8: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("transaction_count")
C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_24040\1013134287.py:22: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("transaction_count")
C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_24040\1013134287.py:36: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("transaction_count")


In [9]:
# Verifying dataset size
print(transactions_df.shape)

(50000, 7)
